[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/05_ood_eval.ipynb)

# Notebook 5 — Out-of-Distribution Eval (k=6–10)

Evaluates the fine-tuned model on StepGame k=6 through k=10 — hop levels **never seen during training**.

Training data covered k=1–5 only. If accuracy on k=6–10 remains above random chance (12.5%), the model generalised spatial reasoning rather than memorised training examples.

Saves results to `results/finetuned/scores_k6_10.json` and a comparison chart.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import ensure_notebook_requirements, prepare_notebook, publish_artifacts, require_local_adapter

REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
print(f'Repo ready at {REPO}')

In [ ]:
ensure_notebook_requirements('notebook04')  # same deps as notebook 04

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

from src.dataset import extract_answer, format_prompt
from src.eval import evaluate, save_results

In [ ]:
ADAPTER_PATH = require_local_adapter(REPO)
OOD_EVAL_PATH = PATHS['data_root'] / 'eval' / 'stepgame_k6_10_filtered.json'
OUT_PATH = PATHS['results_root'] / 'finetuned' / 'predictions_k6_10.json'
SCORES_PATH = PATHS['results_root'] / 'finetuned' / 'scores_k6_10.json'
MAX_SEQ_LENGTH = 1024  # longer context for k=6–10 stories
MAX_NEW_TOKENS = 16
BATCH_SIZE = 4

print(f'OOD eval data: {OOD_EVAL_PATH}')
print(f'Adapter: {ADAPTER_PATH}')

with open(OOD_EVAL_PATH) as f:
    examples_raw = json.load(f)

from collections import Counter
k_dist = Counter(ex['k'] for ex in examples_raw)
print(f'Loaded {len(examples_raw)} examples: {dict(sorted(k_dist.items()))}')

In [ ]:
USE_UNSLOTH = False

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = 'left'
    USE_UNSLOTH = True
    print('Unsloth loaded successfully')
except Exception as e:
    print(f'Unsloth unavailable ({e}), falling back to transformers+peft')
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    base_model_id = 'LiquidAI/LFM2-350M'
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    tokenizer.padding_side = 'left'
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    print('transformers+peft loaded')

In [ ]:
predictions = []

for i in tqdm(range(0, len(examples_raw), BATCH_SIZE)):
    batch = examples_raw[i : i + BATCH_SIZE]
    prompts = [format_prompt(ex['story'], ex['question']) for ex in batch]

    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    for ex, output in zip(batch, outputs):
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        predictions.append({
            'story': ex['story'],
            'question': ex['question'],
            'answer': ex['answer'],
            'prediction': generated,
            'k': ex['k'],
        })

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)

ood_results = evaluate(predictions)
save_results(ood_results, SCORES_PATH)

print(f'\nOverall OOD accuracy (k=6–10): {ood_results["accuracy"]:.1%}')
for k in range(6, 11):
    key = f'accuracy_k{k}'
    if key in ood_results:
        n = sum(1 for p in predictions if p['k'] == k)
        print(f'  k={k}: {ood_results[key]:.1%}  (n={n})')

In [ ]:
# Load k=1–5 fine-tuned scores for the full degradation plot
with open(PATHS['results_root'] / 'finetuned' / 'scores.json') as f:
    ft_k1_5 = json.load(f)

k_vals = list(range(1, 11))
scores = []
ns = []
for k in k_vals:
    key = f'accuracy_k{k}'
    if key in ft_k1_5:
        scores.append(ft_k1_5[key])
        ns.append(50)  # k=1–5 eval: 50 per hop
    elif key in ood_results:
        scores.append(ood_results[key])
        ns.append(sum(1 for p in predictions if p['k'] == k))
    else:
        scores.append(None)
        ns.append(0)

# 95% CI (Wald)
cis = [1.96 * (s * (1 - s) / n) ** 0.5 if s is not None and n > 0 else 0
       for s, n in zip(scores, ns)]

valid = [(k, s, ci) for k, s, ci in zip(k_vals, scores, cis) if s is not None]
xs, ys, errs = zip(*valid)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(xs, ys, yerr=errs, capsize=5, color=['#4C72B0'] * 5 + ['#DD8452'] * 5,
       error_kw={'elinewidth': 1.5})
ax.axhline(0.125, color='grey', linestyle='--', linewidth=1, label='Random (12.5%)')
ax.axvline(5.5, color='black', linestyle=':', linewidth=1.5, label='Training boundary (k≤5)')
ax.set_xticks(k_vals[:len(xs)])
ax.set_xticklabels([f'k={k}' for k in xs])
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Fine-tuned Accuracy: In-Distribution (k=1–5) vs OOD (k=6–10)')
ax.legend()

# Shade OOD region
ax.axvspan(5.5, max(xs) + 0.5, alpha=0.05, color='orange', label='OOD region')

plt.tight_layout()
OUT_CHART = PATHS['results_root'] / 'finetuned' / 'degradation_curve.png'
plt.savefig(OUT_CHART, dpi=150)
plt.show()
print(f'Saved to {OUT_CHART}')

In [ ]:
publish_artifacts(
    [
        'results/finetuned/scores_k6_10.json',
        'results/finetuned/degradation_curve.png',
    ],
    'Add OOD eval results k=6-10 [notebook 05]',
    repo_dir=REPO,
)